# Lab: Hyperparameter Tuning with Keras Tuner — worked study copy

Answers prepared for revision with AI assistance. IBM / Skills Network source notices and embedded instructor explanations are retained. This is not a course submission.

Code completion and execution are tracked separately in the [solution progress report](../../../docs/SOLUTION_PROGRESS.md). Training may require a separate environment, external data, compute or provider credentials.

[Original reference](../../../courses/03-keras-and-tensorflow/notebooks/lab-hyperparameter-tuning-with-keras-tuner.ipynb) · [Course notes](../../../courses/03-keras-and-tensorflow/README.md)

**BOUNDED SMOKE RUN:** seed 42; CSVs sampled to at most 5,000 rows; one grid setting with two folds; training budgets capped; DataLoaders limited to 12 batches. These outputs are execution checks, not full-run model evaluations. Bundled TensorFlow datasets use 512 training / 256 test examples; fit inputs use at most 512 examples or eight dataset batches.

In [1]:
import os, itertools, random
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['TF_NUM_INTEROP_THREADS'] = '1'
os.environ['TF_NUM_INTRAOP_THREADS'] = '1'
import numpy as np
np.random.seed(42)
random.seed(42)
import pandas as pd
_read_csv = pd.read_csv
def _bounded_csv(*args, **kwargs):
    data = _read_csv(*args, **kwargs)
    return data.sample(n=5000,random_state=42) if len(data)>5000 else data
pd.read_csv = _bounded_csv
from sklearn.model_selection import GridSearchCV
_grid_fit = GridSearchCV.fit
def _bounded_grid(self, *args, **kwargs):
    grids = self.param_grid if isinstance(self.param_grid,list) else [self.param_grid]
    self.param_grid = [{k:list(v)[:1] for k,v in grid.items()} for grid in grids[:1]]
    self.cv = 2
    self.n_jobs = 1
    self.verbose = 0
    return _grid_fit(self,*args,**kwargs)
GridSearchCV.fit = _bounded_grid
try:
    import torch
except ImportError:
    torch = None
if torch is not None:
    torch.manual_seed(42)
    torch.set_num_threads(1)
    _loader_iter = torch.utils.data.DataLoader.__iter__
    def _bounded_loader(self):
        return itertools.islice(_loader_iter(self),12)
    torch.utils.data.DataLoader.__iter__ = _bounded_loader
import tensorflow as tf
tf.random.set_seed(42)
for _dataset in [tf.keras.datasets.mnist,tf.keras.datasets.fashion_mnist,tf.keras.datasets.cifar10]:
    _original_load = _dataset.load_data
    def _small_load(*args,_load=_original_load,**kwargs):
        (x,y),(xt,yt) = _load(*args,**kwargs)
        return (x[:512],y[:512]),(xt[:256],yt[:256])
    _dataset.load_data = _small_load
_keras_fit = tf.keras.Model.fit
def _small_fit(self,*args,**kwargs):
    args = list(args)
    def _limit(value):
        if isinstance(value,(np.ndarray,pd.DataFrame,pd.Series)):return value[:512]
        if isinstance(value,(tuple,list)) and value and all(isinstance(x,np.ndarray) for x in value):return type(value)(x[:512] for x in value)
        if isinstance(value,tf.data.Dataset):return value.take(8)
        return value
    for i in range(min(2,len(args))):args[i]=_limit(args[i])
    for key in ['x','y','validation_data']:
        if key in kwargs:kwargs[key]=_limit(kwargs[key])
    kwargs['epochs']=min(2,kwargs.get('epochs',2))
    data=args[0] if args else kwargs.get('x')
    if hasattr(data,'class_indices'):
        kwargs['steps_per_epoch']=min(2,len(data))
        if hasattr(kwargs.get('validation_data'),'class_indices'):
            kwargs['validation_steps']=min(2,len(kwargs['validation_data']))
    return _keras_fit(self,*args,**kwargs)
tf.keras.Model.fit = _small_fit


Matplotlib is building the font cache; this may take a moment.


<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Lab: Hyperparameter Tuning with Keras Tuner**


Estimated time needed: **30** minutes


In this lab, you will learn how to set up Keras Tuner and prepare the environment for hyperparameter tuning. 

## Learning objectives: 
By the end of this lab, you will: 
- Install Keras Tuner and import the necessary libraries
- Load and preprocess the MNIST data set
- Define a model-building function that uses hyperparameters to configure the model architecture
- Set up Keras Tuner to search for the best hyperparameter configuration 
- Retrieve the best hyperparameters from the search and build a model with these optimized values

## Prerequisites: 
- Basic understanding of Python programming 
- Keras and TensorFlow installed


### Exercise 1: Install the Keras Tuner 

This exercise guides you through the initial setup for using Keras Tuner. You install the library, import necessary modules, and load and preprocess the MNIST data set, which will be used for hyperparameter tuning. 
1. **Install Keras Tuner:**
    - Use pip to install Keras Tuner
2. **Import necessary libraries:**
    - Import Keras Tuner, TensorFlow, and Keras modules
3. **Load and preprocess the MNIST data set:**
    - Load the MNIST data set.
    - Normalize the data set by dividing by 255.0.


#### Explanation: 
This code installs the necessary libraries using pip

- **TensorFlow**: Ensures compatibility with the Keras Tuner.
- **Keras Tuner**: The version used in this lab.
- **Numpy**: Ensures compatibility with the other installed packages.


In [2]:
import sys
sys.setrecursionlimit(100000)

#### Explanation: 
The sys.setrecursionlimit function is used to increase the recursion limit, which helps prevent potential recursion errors when running complex models with deep nested functions or when using certain libraries like TensorFlow.


In [3]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

#### Explanation

This code imports the necessary libraries:

- **`keras_tuner`**: Used for hyperparameter tuning.
- **`Sequential`**: A linear stack of layers in Keras.
- **`Dense`**, **`Flatten`**: Common Keras layers.
- **`mnist`**: The MNIST dataset, a standard dataset for image classification.
- **`Adam`**: An optimizer in Keras.


In [4]:
(x_train, y_train), (x_val, y_val) = mnist.load_data()
x_train, x_val = (x_train / 255.0, x_val / 255.0)
print(f'Training data shape: {x_train.shape}')
print(f'Validation data shape: {x_val.shape}')

Training data shape: (512, 28, 28)
Validation data shape: (256, 28, 28)


#### Explanation

This code loads the MNIST dataset and preprocesses it:

- **`mnist.load_data()`**: Loads the dataset, returning training and validation splits.
- **`x_train / 255.0`**: Normalizes the pixel values to be between 0 and 1.
- **`print(f'...')`**: Displays the shapes of the training and validation datasets.


### Exercise 2: Defining the model with hyperparameters 

In this exercise, you define a model-building function that uses the `HyperParameters` object to specify the number of units in a dense layer and the learning rate. This function returns a compiled Keras model that is ready for hyperparameter tuning.

**Define a model-building function:**
- Create a function `build_model` that takes a `HyperParameters` object as input.
- Use the `HyperParameters` object to define the number of units in a dense layer and the learning rate for the optimizer.
- Compile the model with sparse categorical cross-entropy loss and Adam optimizer.


In [5]:
def build_model(hp):
    model = Sequential([Flatten(input_shape=(28, 28)), Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'), Dense(10, activation='softmax')])
    model.compile(optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=0.0001, max_value=0.01, sampling='LOG')), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

#### Explanation

This function builds and compiles a Keras model with hyperparameters:

- **`hp.Int('units', ...)`**: Defines the number of units in the Dense layer as a hyperparameter.
- **`hp.Float('learning_rate', ...)`**: Defines the learning rate as a hyperparameter.
- **`model.compile()`**: Compiles the model with the Adam optimizer and sparse categorical cross-entropy loss.


### Exercise 3: Configuring the hyperparameter search 

This exercise guides you through configuring Keras Tuner. You create a `RandomSearch` tuner, specifying the model-building function, the optimization objective, the number of trials, and the directory for storing results. The search space summary provides an overview of the hyperparameters being tuned. 

**Create a RandomSearch Tuner:**
- Use the `RandomSearch` class from Keras Tuner. 
- Specify the model-building function, optimization objective (validation accuracy), number of trials, and directory for storing results.


In [6]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=10, executions_per_trial=2, directory='my_dir', project_name='intro_to_kt')
tuner.search_space_summary()

Search space summary
Default search space size: 2
units (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


#### Explanation

This code sets up a Keras Tuner `RandomSearch`:

- **`build_model`**: The model-building function.
- **`objective='val_accuracy'`**: The metric to optimize (validation accuracy).
- **`max_trials=10`**: The maximum number of different hyperparameter configurations to try.
- **`executions_per_trial=2`**: The number of times to run each configuration.
- **`directory='my_dir'`**: Directory to save the results.
- **`project_name='intro_to_kt'`**: Name of the project for organizing results.

Displays a summary of the hyperparameter search space, providing an overview of the hyperparameters being tuned.



### Exercise 4: Running the hyperparameter search 

In this exercise, you run the hyperparameter search using the `search` method of the tuner. You provide the training and validation data along with the number of epochs. After the search is complete, the results summary displays the best hyperparameter configurations found. 

**Run the search:**
- Use the `search` method of the tuner. 
- Pass in the training data, validation data, and the number of epochs


In [7]:
tuner.search(x_train, y_train, epochs=2, validation_data=(x_val, y_val))
tuner.results_summary()

Trial 10 Complete [00h 00m 01s]
val_accuracy: 0.787109375

Best val_accuracy So Far: 0.849609375
Total elapsed time: 00h 00m 11s
Results summary
Results in my_dir/intro_to_kt
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 00 summary
Hyperparameters:
units: 448
learning_rate: 0.0033276572881909957
Score: 0.849609375

Trial 04 summary
Hyperparameters:
units: 448
learning_rate: 0.0005935157243286463
Score: 0.8046875

Trial 02 summary
Hyperparameters:
units: 192
learning_rate: 0.0017377012853110221
Score: 0.802734375

Trial 03 summary
Hyperparameters:
units: 288
learning_rate: 0.000978041751919679
Score: 0.796875

Trial 01 summary
Hyperparameters:
units: 160
learning_rate: 0.002051386718289359
Score: 0.794921875

Trial 09 summary
Hyperparameters:
units: 224
learning_rate: 0.0010992362868788867
Score: 0.787109375

Trial 07 summary
Hyperparameters:
units: 32
learning_rate: 0.004114773318984142
Score: 0.78125

Trial 05 summary
Hyperparameters:
units: 416
learnin

#### Explanation

This command runs the hyperparameter search:

- **`epochs=5`**: Each trial is trained for 5 epochs.
- **`validation_data=(x_val, y_val)`**: The validation data to evaluate the model's performance during the search.

After the search is complete, this command displays a summary of the best hyperparameter configurations found during the search.


## Exercise 5: Analyzing and using the best hyperparameters 

In this exercise, you retrieve the best hyperparameters found during the search and print their values. You then build a model with these optimized hyperparameters and train it on the full training data set. Finally, you evaluate the model’s performance on the test set to ensure that it performs well with the selected hyperparameters. 

**Retrieve the best hyperparameters:**
- Use the `get_best_hyperparameters` method to get the best hyperparameters. 
- Print the optimal values for the hyperparameters. 

**Build and train the model:**
- Build a model using the best hyperparameters. 
- Train the model on the full training data set and evaluate its performance on the test set.


In [8]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f" \n\nThe optimal number of units in the first dense layer is {best_hps.get('units')}. \n\nThe optimal learning rate for the optimizer is {best_hps.get('learning_rate')}. \n\n")
model = tuner.hypermodel.build(best_hps)
model.fit(x_train, y_train, epochs=2, validation_split=0.2)
test_loss, test_acc = model.evaluate(x_val, y_val)
print(f'Test accuracy: {test_acc}')

 

The optimal number of units in the first dense layer is 448. 

The optimal learning rate for the optimizer is 0.0033276572881909957. 


Epoch 1/2


 1/13 ━━━━━━━━━━━━━━━━━━━━ 2s 217ms/step - accuracy: 0.1250 - loss: 2.3441

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5090 - loss: 1.5648 - val_accuracy: 0.7184 - val_loss: 0.9058


Epoch 2/2


 1/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8750 - loss: 0.3518

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9027 - loss: 0.3101 - val_accuracy: 0.7864 - val_loss: 0.7571


1/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8750 - loss: 0.5892

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8190 - loss: 0.6005


Test accuracy: 0.80859375


#### Explanation

This code retrieves the best hyperparameters found during the search:

- **`get_best_hyperparameters(num_trials=1)`**: Gets the best hyperparameter configuration.
- **`print(f"...")`**: Prints the best hyperparameters.
- **`model.fit(...)`**: Trains the model on the full training data with a validation split of 20%.
- **`model.evaluate(...)`**: Evaluates the model on the test (validation) dataset and prints the accuracy, which gives an indication of how well the model generalizes.


## Practice exercises 

### Exercise 1: Setting Up Keras Tuner 

#### Objective: 
Learn how to set up Keras Tuner and prepare the environment for hyperparameter tuning. 

#### Instructions: 
1. Install Keras Tuner.
2. Import necessary libraries.
3. Load and preprocess the MNIST data set.


In [9]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam
(x_train, y_train), (x_val, y_val) = mnist.load_data()
x_train, x_val = (x_train / 255.0, x_val / 255.0)
print(f'Training data shape: {x_train.shape}')
print(f'Validation data shape: {x_val.shape}')

Training data shape: (512, 28, 28)
Validation data shape: (256, 28, 28)


<details>
    <summary>Click here for Solution</summary>

```python
!pip install keras-tuner 

# Step 2: Import necessary libraries 
import keras_tuner as kt 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense, Flatten 
from tensorflow.keras.datasets import mnist 
from tensorflow.keras.optimizers import Adam 

# Step 3: Load and preprocess the MNIST data set 
(x_train, y_train), (x_val, y_val) = mnist.load_data() 
x_train, x_val = x_train / 255.0, x_val / 255.0 

# Print the shapes of the training and validation datasets
print(f'Training data shape: {x_train.shape}') 
print(f'Validation data shape: {x_val.shape}')
```

</details>


### Exercise 2: Defining the model with hyperparameters 

#### Objective: 
Define a model-building function that uses hyperparameters to configure the model architecture. 

#### Instructions: 
1. Define a model-building function that uses the `HyperParameters` object to specify the number of units in a dense layer and the learning rate. 
2. Compile the model with sparse categorical cross-entropy loss and Adam optimizer. 


In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
import keras_tuner as kt

def build_model(hp):
    model = Sequential([Flatten(input_shape=(28, 28)), Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'), Dense(10, activation='softmax')])
    model.compile(optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=0.0001, max_value=0.01, sampling='LOG')), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

<details>
    <summary>Click here for Solution</summary>

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
import keras_tuner as kt

# Step 1: Define a model-building function
def build_model(hp):
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model
```

</details>


### Exercise 3: Configuring the hyperparameter search 

#### Objective: 
Set up Keras Tuner to search for the best hyperparameter configuration. 

#### Instructions: 
1. Create a `RandomSearch` tuner using the model-building function. 
2. Specify the optimization objective, number of trials, and directory for storing results.


In [11]:
import keras_tuner as kt
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=10, executions_per_trial=2, directory='my_dir', project_name='intro_to_kt')
tuner.search_space_summary()

Reloading Tuner from my_dir/intro_to_kt/tuner0.json
Search space summary
Default search space size: 2
units (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


<details>
    <summary>Click here for Solution</summary>

```python
import keras_tuner as kt

# Step 1: Create a RandomSearch Tuner
tuner = kt.RandomSearch(
    build_model,  # Ensure 'build_model' function is defined from previous code
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='intro_to_kt'
)

# Display a summary of the search space
tuner.search_space_summary()
```

</details>


### Exercise 4: Running the hyperparameter search

#### Objective: 
Run the hyperparameter search and dispaly the summary of the results.

#### Instructions: 
1. Run the hyperparameter search using the `search` method of the tuner. 
2. Pass in the training data, validation data, and the number of epochs. 
3. Display a summary of the results. 


In [12]:
tuner.search(x_train, y_train, epochs=2, validation_data=(x_val, y_val))
tuner.results_summary()

Results summary
Results in my_dir/intro_to_kt
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 00 summary
Hyperparameters:
units: 448
learning_rate: 0.0033276572881909957
Score: 0.849609375

Trial 04 summary
Hyperparameters:
units: 448
learning_rate: 0.0005935157243286463
Score: 0.8046875

Trial 02 summary
Hyperparameters:
units: 192
learning_rate: 0.0017377012853110221
Score: 0.802734375

Trial 03 summary
Hyperparameters:
units: 288
learning_rate: 0.000978041751919679
Score: 0.796875

Trial 01 summary
Hyperparameters:
units: 160
learning_rate: 0.002051386718289359
Score: 0.794921875

Trial 09 summary
Hyperparameters:
units: 224
learning_rate: 0.0010992362868788867
Score: 0.787109375

Trial 07 summary
Hyperparameters:
units: 32
learning_rate: 0.004114773318984142
Score: 0.78125

Trial 05 summary
Hyperparameters:
units: 416
learning_rate: 0.000263474572751995
Score: 0.67578125

Trial 08 summary
Hyperparameters:
units: 352
learning_rate: 0.0002160113604403082

<details>
    <summary>Click here for Solution</summary>

```python
# Step 1: Run the hyperparameter search 

tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val)) 

 # Display a summary of the results 

tuner.results_summary()
```

</details>


### Exercise 5: Analyzing and using the best hyperparameters 

#### Objective: 
Retrieve the best hyperparameters from the search and build a model with these optimized values. 

#### Instructions: 
1. Retrieve the best hyperparameters using the `get_best_hyperparameters` method. 
2. Build a model using the best hyperparameters. 
3. Train the model on the full training data set and evaluate its performance on the validation set.


In [13]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f" \n\nThe optimal number of units in the first dense layer is {best_hps.get('units')}. \n\nThe optimal learning rate for the optimizer is {best_hps.get('learning_rate')}. \n\n")
model = tuner.hypermodel.build(best_hps)
model.fit(x_train, y_train, epochs=2, validation_split=0.2)
val_loss, val_acc = model.evaluate(x_val, y_val)
print(f'Validation accuracy: {val_acc}')

 

The optimal number of units in the first dense layer is 448. 

The optimal learning rate for the optimizer is 0.0033276572881909957. 




Epoch 1/2


 1/13 ━━━━━━━━━━━━━━━━━━━━ 2s 207ms/step - accuracy: 0.0312 - loss: 2.3209

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4813 - loss: 1.4933 - val_accuracy: 0.7282 - val_loss: 0.8078


Epoch 2/2


 1/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8750 - loss: 0.3961

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9044 - loss: 0.3161 - val_accuracy: 0.8155 - val_loss: 0.7557


1/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8750 - loss: 0.6505

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8107 - loss: 0.6051


Validation accuracy: 0.80859375


<details>
    <summary>Click here for Solution</summary>

```python
# Step 1: Retrieve the best hyperparameters 

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0] 

print(f""" 

The optimal number of units in the first dense layer is {best_hps.get('units')}. 

The optimal learning rate for the optimizer is {best_hps.get('learning_rate')}. 

""") 

 # Step 2: Build and train the model with best hyperparameters 

model = tuner.hypermodel.build(best_hps) 

model.fit(x_train, y_train, epochs=10, validation_split=0.2) 

 # Evaluate the model on the validation set 

val_loss, val_acc = model.evaluate(x_val, y_val) 

print(f'Validation accuracy: {val_acc}') 
```

</details>


### Conclusion 

Congratulations on completing this lab! You have learned to set up Keras Tuner and prepare the environment for hyperparameter tuning. In addition, you defined a model-building function that uses hyperparameters to configure the model architecture. You configured Keras Tuner to search for the best hyperparameter configuration and learned to run the hyperparameter search and analyze the results. Finally, you retrieved the best hyperparameters and built a model with these optimized values. 


## Authors


Skillup


Copyright ©IBM Corporation. All rights reserved.
